In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


In [3]:
df = pd.read_csv("mental_health_dataset.csv")
df.head()

,age,gender,employment_status,work_environment,mental_health_history,seeks_treatment,stress_level,sleep_hours,physical_activity_days,depression_score,anxiety_score,social_support_score,productivity_score,mental_health_risk
0,56,Male,Employed,On-site,Yes,Yes,6,6.2,3,28,17,54,59.7,High
1,46,Female,Student,On-site,No,Yes,10,9.0,4,30,11,85,54.9,High
2,32,Female,Employed,On-site,Yes,No,7,7.7,2,24,7,62,61.3,Medium
3,60,Non-binary,Self-employed,On-site,No,No,4,4.5,4,6,0,95,97.0,Low
4,25,Female,Self-employed,On-site,Yes,Yes,3,5.4,0,24,12,70,69.0,High


### Load the dataset

Here, we load the dataset from a CSV file into a pandas DataFrame.

This allows us to:
- inspect the data
- access columns easily
- prepare the data for machine learning

Displaying the first rows helps verify that the file was loaded correctly.


In [4]:
target_col = "mental_health_risk"

X = df.drop(columns=[target_col])
y = df[target_col]

num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = X.select_dtypes(include=["object", "bool"]).columns.tolist()

num_cols, cat_cols


(['age',
  'stress_level',
  'sleep_hours',
  'physical_activity_days',
  'depression_score',
  'anxiety_score',
  'social_support_score',
  'productivity_score'],
 ['gender',
  'employment_status',
  'work_environment',
  'mental_health_history',
  'seeks_treatment'])

### Define target and features

In this block:
- We separate the **target variable** (`mental_health_risk`) from the input features.
- `X` contains all the features.
- `y` contains the values we want to predict.

We also separate:
- numerical features (numbers)
- categorical features (text values)

This separation is necessary because they are preprocessed differently.


In [5]:
preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ],
    remainder="drop"
)


### Preprocessing pipeline

Here we define how the data will be prepared before training the model.

- Numerical features are standardized so they are on the same scale.
- Categorical features are converted into numbers using one-hot encoding.
- This step is mandatory because machine learning models only work with numbers.

The preprocessing is applied automatically inside the pipeline.


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)


### Train / test split

We split the dataset into two parts:
- a training set (75%) used to train the model
- a test set (25%) used to evaluate performance on unseen data

Stratification ensures that each class of the target variable
is well represented in both sets.


In [7]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


### Cross-validation

Cross-validation is used to get a more reliable evaluation of the model.

The training set is split into several folds:
- the model is trained multiple times
- each time on a different subset of the data

This reduces the risk of random results.


In [9]:
base_clf = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("clf", LogisticRegression(max_iter=5000)),
    ]
)


### Baseline model (no PCA)

This block defines the baseline classifier.

- We use logistic regression as a simple and interpretable model.
- The model is trained using all original features.
- No dimensionality reduction is applied here.

This baseline will be used as a reference to evaluate PCA later.


In [10]:
baseline_cv_scores = cross_val_score(
    base_clf,
    X_train,
    y_train,
    cv=cv,
    scoring="accuracy"
)

base_clf.fit(X_train, y_train)
baseline_test_acc = accuracy_score(y_test, base_clf.predict(X_test))

print("=== Baseline (no PCA) ===")
print(f"CV accuracy: {baseline_cv_scores.mean():.4f} ± {baseline_cv_scores.std():.4f}")
print(f"Test accuracy: {baseline_test_acc:.4f}")


=== Baseline (no PCA) ===
CV accuracy: 0.9968 ± 0.0011
Test accuracy: 0.9988


### Baseline evaluation

Here we evaluate the baseline model.

- Cross-validation accuracy gives a stable estimate of performance.
- Test accuracy measures how well the model works on new data.

These results will be compared with the PCA-based models.


In [11]:
Xt_train_pre = preprocess.fit_transform(X_train)
Xt_test_pre = preprocess.transform(X_test)

max_components = min(60, Xt_train_pre.shape[1])
max_components


23

### Prepare data for PCA

In this block, we apply the preprocessing steps separately.

This is needed to:
- know how many numerical features we have after encoding
- compute the maximum possible number of PCA components

PCA can only be applied after standardization.


In [12]:
components_list = list(range(1, max_components + 1))

cv_means = []
cv_stds = []
test_accs = []
explained_var = []


### PCA experiment setup

We prepare a list of PCA dimensions to test.

For each number of components, we will:
- train a classifier
- evaluate its performance
- measure how much information is kept

We also initialize lists to store the results.


In [14]:
for k in components_list:
    pipe = Pipeline(
        steps=[
            ("preprocess", preprocess),
            ("pca", PCA(n_components=k, random_state=42)),
            ("clf", LogisticRegression(max_iter=5000)),
        ]
    )

    scores = cross_val_score(
        pipe,
        X_train,
        y_train,
        cv=cv,
        scoring="accuracy"
    )
    cv_means.append(scores.mean())
    cv_stds.append(scores.std())

    pipe.fit(X_train, y_train)
    test_accs.append(
        accuracy_score(y_test, pipe.predict(X_test))
    )

    pca = PCA(n_components=k, random_state=42)
    pca.fit(Xt_train_pre)
    explained_var.append(
        pca.explained_variance_ratio_.sum()
    )


### PCA and classification loop

This is the main experiment.

For each number of PCA components:
- the data is preprocessed
- dimensionality is reduced using PCA
- a logistic regression model is trained
- accuracy is measured using cross-validation and test data

This allows us to study how reducing the number of features
affects classification performance.


In [15]:
results = pd.DataFrame(
    {
        "n_components": components_list,
        "cv_accuracy_mean": cv_means,
        "cv_accuracy_std": cv_stds,
        "test_accuracy": test_accs,
        "explained_variance_ratio_sum": explained_var,
    }
)

results.head(10)


,n_components,cv_accuracy_mean,cv_accuracy_std,test_accuracy,explained_variance_ratio_sum
0,1,0.689200,0.008234,0.6896,0.181101
1,2,0.722267,0.029387,0.7364,0.277984
2,3,0.753067,0.034529,0.7820,0.373636
3,4,0.820533,0.017171,0.8520,0.468718
4,5,0.835333,0.010031,0.8632,0.561028
5,6,0.850000,0.015816,0.8640,0.652411
6,7,0.936533,0.007123,0.9520,0.741672
7,8,0.936533,0.008224,0.9512,0.786558
8,9,0.936667,0.007922,0.9512,0.828856
9,10,0.937467,0.007971,0.9516,0.867949


### Results table

All results are gathered into a DataFrame.

For each number of PCA components, we store:
- mean cross-validation accuracy
- accuracy standard deviation
- test accuracy
- explained variance ratio

This table makes it easy to analyze and compare models.


In [16]:
best_idx = int(np.argmax(results["cv_accuracy_mean"].values))
best_row = results.iloc[best_idx]

print("=== Best by CV mean accuracy ===")
print(best_row)


=== Best by CV mean accuracy ===
n_components                    18.000000
cv_accuracy_mean                 0.996933
cv_accuracy_std                  0.001373
test_accuracy                    0.999200
explained_variance_ratio_sum     1.000000
Name: 17, dtype: float64


### Best PCA configuration

Here we identify the PCA model with the highest average
cross-validation accuracy.

This gives the best trade-off between:
- dimensionality reduction
- prediction performance


In [17]:
plateau_threshold = baseline_cv_scores.mean() - 0.01
plateau_candidates = results[results["cv_accuracy_mean"] >= plateau_threshold]

if len(plateau_candidates) > 0:
    smallest_plateau = plateau_candidates.iloc[0]
    print("=== Smallest PCA dimension within 1% of baseline CV accuracy ===")
    print(smallest_plateau)
else:
    print("No PCA dimension reached within 1% of baseline CV accuracy.")


=== Smallest PCA dimension within 1% of baseline CV accuracy ===
n_components                    17.000000
cv_accuracy_mean                 0.995467
cv_accuracy_std                  0.001147
test_accuracy                    0.997600
explained_variance_ratio_sum     0.995376
Name: 16, dtype: float64


### PCA vs baseline comparison

We check if a reduced number of PCA components
can achieve nearly the same performance as the baseline model.

If this is the case, dimensionality reduction is justified
because we obtain similar accuracy with fewer features.


In [18]:
results.to_csv("pca_supervised_performance_results.csv", index=False)
print("Saved: pca_supervised_performance_results.csv")


Saved: pca_supervised_performance_results.csv
